In [2]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

TAREA: Los ejemplos ilustrativos anteriores permiten saber el número de monedas presentes en la imagen. ¿Cómo saber la cantidad de dinero presente en ella? Sugerimos identificar de forma interactiva (por ejemplo haciendo clic en la imagen) una moneda de un valor determinado en la imagen (por ejemplo de 1€). Tras obtener esa información y las dimensiones en milímetros de las distintas monedas, realiza una propuesta para estimar la cantidad de dinero en la imagen. Muestra la cuenta de monedas y dinero sobre la imagen. No hay restricciones sobre utilizar medidas geométricas o de color. 

Una vez resuelto el reto con la imagen ideal proporcionada, captura una o varias imágenes con monedas. Aplica el mismo esquema, tras identificar la moneda del valor determinado, calcula el dinero presente en la imagen. ¿Funciona correctamente? ¿Se observan problemas?

Nota: Para establecer la correspondencia entre píxeles y milímetros, comentar que la moneda de un euro tiene un diámetro de 23.25 mm. la de 50 céntimos de 24.35, la de 20 céntimos de 22.25, etc. 

Extras: Considerar que la imagen pueda contener objetos que no son monedas y/o haya solape entre las monedas. Demo en vivo. 



In [ ]:
# leemos la imagen
img = cv2.imread('Monedas.jpg')

# hago la imagen más pequeña para que quepa en la pantalla
escala = 0.5
ancho = int(img.shape[1] * escala)
alto = int(img.shape[0] * escala)
img = cv2.resize(img, (ancho, alto), interpolation=cv2.INTER_AREA)

# paso la imagen a grises
img_gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# se establece el umbral
umbral = 200
th1,img_th1 = cv2.threshold(img_gris,umbral,255,cv2.THRESH_BINARY_INV)

# se obtienen los contornos externos
contornos, hierarchy = cv2.findContours(img_th1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

#dibujamos estos contornos externos
img_contornos = img.copy()
cv2.drawContours(img_contornos, contornos, -1, (0, 255, 0), 2)

# Variable para el tamaño de la de un euro
tamaño_moneda_euro = None

def calcular_dinero():
    # aquí se calcula el dinero en función del tamaño de las monedas
    if tamaño_moneda_euro == None:
        print("Primero haz click en la moneda de un euro.")
        return
    
    numero_monedas = 0
    dinero = 0

    # Recorremos todos los contornos
    for c in contornos:
        #Área del contorno
        area = cv2.contourArea(c)
        if area > 10:
            relacion = round(area / tamaño_moneda_euro, 2)
            
            if 0.95 <= relacion <= 1.05:
                dinero += 1
            elif relacion <= 0.5: # Aproximadamente 1 céntimo
                dinero += 0.01
            elif 0.65 <= relacion <= 0.69: # Aproximadamente 2 céntimos
                dinero += 0.02
            elif 0.8 <= relacion <= 0.84: # Aproximadamente 5 céntimos
                dinero += 0.05
            elif 0.7 <= relacion <= 0.74: # Aproximadamente 10 céntimos
                dinero += 0.10
            elif 0.9 <= relacion <= 0.95: # Aproximadamente 20 céntimos
                dinero += 0.20
            elif 1.05 <= relacion <= 1.11: # Aproximadamente 50 céntimos
                dinero += 0.50
            elif relacion >= 1.24: # Aproximadamente 2 euros
                dinero += 2.00

            numero_monedas += 1

    print(f"Número de monedas detectadas: {numero_monedas}")
    print(f"Dinero total estimado: {dinero}")

def evento_click(event, x, y, flags, param):
    # para manejar el evento al hacer click
    global tamaño_moneda_euro

    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"Click en coordenadas: {x}, {y}")

        # Buscamos el contorno
        for c in contornos:
            result = cv2.pointPolygonTest(c, (x, y), False)
            if result >= 0:
                area = cv2.contourArea(c)
                tamaño_moneda_euro = area
                print(f"Moneda de un euro seleccionada, su área es: {area}")
            
                calcular_dinero()
                break
        

# creo una ventana para poder detectar el click, asigno el evento y muestro la imagen
cv2.namedWindow("Detector de monedas")
cv2.setMouseCallback("Detector de monedas", evento_click)
cv2.imshow('Detector de monedas', img_contornos)

# esperamos a que se pulse una tecla y cerramos
cv2.waitKey(0)
cv2.destroyAllWindows();

Click en coordenadas: 76, 192
Moneda de un euro seleccionada, su área es: 6643.0
Número de monedas detectadas:  8
Dinero total estimado: 3.88
Click en coordenadas: 76, 192
Moneda de un euro seleccionada, su área es: 6643.0
Número de monedas detectadas:  8
Dinero total estimado: 3.88
